# 01 – Data Exploration

This notebook performs an initial exploration of all five CSV files from the
[NBA Games Dataset](https://www.kaggle.com/datasets/nathanlauga/nba-games) on Kaggle.

**Goals:**
- Load each CSV and inspect shape, dtypes, and sample rows
- Identify missing values
- Visualise score distributions and win rates

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'NBA-STATS'))
# If running from notebooks/ directory:
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_games, load_games_details, load_players, load_ranking, load_teams

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Datasets

In [ ]:
RAW_DIR = os.path.join(os.path.abspath('..'), 'data', 'raw')

games = load_games(RAW_DIR)
details = load_games_details(RAW_DIR)
players = load_players(RAW_DIR)
ranking = load_ranking(RAW_DIR)
teams = load_teams(RAW_DIR)

print('games      :', games.shape)
print('details    :', details.shape)
print('players    :', players.shape)
print('ranking    :', ranking.shape)
print('teams      :', teams.shape)

## 2. Inspect Each Dataset

In [ ]:
games.head()

In [ ]:
games.info()

In [ ]:
games.describe()

In [ ]:
details.head()

In [ ]:
players.head()

In [ ]:
teams.head()

## 3. Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title) in zip(axes, [(games, 'games'), (details, 'games_details')]):
    missing = df.isnull().mean().sort_values(ascending=False)
    missing = missing[missing > 0]
    if missing.empty:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14)
    else:
        missing.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_ylabel('Missing fraction')
    ax.set_title(f'Missing Values — {title}')

plt.tight_layout()
plt.show()

## 4. Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(games['PTS_home'], bins=30, alpha=0.6, label='Home', color='royalblue')
axes[0].hist(games['PTS_away'], bins=30, alpha=0.6, label='Away', color='tomato')
axes[0].set_xlabel('Points Scored')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Points Scored')
axes[0].legend()

home_win_rate = games['HOME_TEAM_WINS'].mean()
axes[1].bar(['Home Wins', 'Away Wins'], [home_win_rate, 1 - home_win_rate],
            color=['royalblue', 'tomato'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Win Rate')
axes[1].set_title(f'Home vs Away Win Rate (home = {home_win_rate:.1%})')

plt.tight_layout()
plt.show()

## 5. Games Per Season

In [ ]:
if 'SEASON' in games.columns:
    season_counts = games.groupby('SEASON').size()
    season_counts.plot(kind='bar', figsize=(12, 4), color='steelblue')
    plt.xlabel('Season')
    plt.ylabel('Number of Games')
    plt.title('Games Per Season')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No SEASON column found in games.csv')

## 6. Top Teams by Win Rate

In [ ]:
# Count wins for each team (home wins + away wins)
home_wins = games[games['HOME_TEAM_WINS'] == 1].groupby('HOME_TEAM_ID').size().rename('wins')
home_games = games.groupby('HOME_TEAM_ID').size().rename('games')
away_wins = games[games['HOME_TEAM_WINS'] == 0].groupby('VISITOR_TEAM_ID').size().rename('wins')
away_games = games.groupby('VISITOR_TEAM_ID').size().rename('games')

team_wins = (home_wins.add(away_wins, fill_value=0)).astype(int)
team_games = (home_games.add(away_games, fill_value=0)).astype(int)
win_rate = (team_wins / team_games).sort_values(ascending=False)

# Merge with team names if available
if 'TEAM_ID' in teams.columns and 'NICKNAME' in teams.columns:
    name_map = teams.set_index('TEAM_ID')['NICKNAME']
    win_rate.index = win_rate.index.map(lambda tid: name_map.get(tid, str(tid)))

win_rate.head(15).plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.xlabel('Win Rate')
plt.title('Top 15 Teams by Win Rate (all seasons)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()